# run_circuit：运行电路与分析结果参考

本 notebook 是代码示例，生成时没有执行或测试任何单元。项目实际函数名是 `qcontrol.experiment.circuits.run_circuit`（单数）。

内容：实验配置 → 执行 QASM 电路 → 获取概率表 → 绘图与关联量 → 从磁盘或 Data Vault 读回 → 导出。默认采用二态读取、四比特、Z 基测量。硬件配置、比特映射和数据目录需要按实验现场调整。

原始数据流程：每次重复测量得到 IQ，按读出标定判态，再统计联合概率。返回和保存的是概率表；原始 IQ 和逐次测量 bitstring 没有保存在该表中。

## 0. 当前源码的使用前提

当前 `qcontrol/qcontrol/experiment/circuits.py` 中的内部回调是：

```python
def circuit_func(device_configs, circuit, theta, QST_base):
```

但 `parse_exp_args(...)` 只传入 `circuit` 和 `QST_base`，会导致缺少 `theta` 参数。该回调里的形参 `theta` 没有被使用，最小修正是从签名中删除它：

```python
def circuit_func(device_configs, circuit, QST_base):
```

**本 notebook 没有修改项目源码。下方实验调用以完成此修正为前提。** 在外层调用中添加 `theta=...` 不能解决问题，因为该参数没有进入扫描器。

本示例显式设置 `axes=["circuit", "QST_base"]`，并使用 `nstate=2`、读出比特数不超过 10，以避开当前列定义不匹配的问题。当前电路执行分支实现了 `rx/rz/cz`；其他门不会由这些分支执行，不应直接套用任意 QASM 文件。

## 1. 分析库和路径

如果只分析历史结果，可以执行本节后直接跳到第 7 节，不需要导入硬件模块。

In [ ]:
from pathlib import Path
from itertools import product
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = Path(r"D:\qcontrol_gd\qcontrol")

## 2. 实验配置

QASM 中的 `q[i]` 对应 `opt_qubits[i]`。下面四比特及耦合器列表参考现有实验 notebook，使用时要与实际电路连接、CZ 标定和接线一致。

`data_vault_path` 是 LabRAD 逻辑目录，不是 Windows 磁盘路径。设备服务及 Data Vault 的启动沿用现有实验环境。

In [ ]:
from importlib import reload

from qcontrol.config import wiring_configs
from qcontrol.utils.qconfig import QConfig
from qcontrol.units import Unit
from qcontrol.experiment import circuits as qc

ns = Unit("ns")
qubit_configs = QConfig(
    str(PROJECT_ROOT / "qcontrol/config/data/N36X21_20260801.json")
)

opt_qubits = ["q11_7", "q11_9", "q9_9", "q9_11"]
read_qubits = list(opt_qubits)
opt_couplers = ["c11_8", "c10_9", "c9_10"]

circuit_paths = [
    PROJECT_ROOT / "circuit" / f"opt_circuit_{i}.qasm"
    for i in range(1, 5)
]
data_vault_path = ["", "N36X21", "20260909_run_circuit_example"]
reps = 3000
measure_base = "Z"
do_qst = False

circuit_manifest = pd.DataFrame({
    "circuit": range(len(circuit_paths)),
    "qasm_path": [str(p) for p in circuit_paths],
})
display(circuit_manifest)

当前项目 notebook 使用如下方式指定设备管理器。已经在本 kernel 初始化过时，可以复用现有对象。

In [ ]:
from qcontrol.experiment import run_seq
from qcontrol.server.device_manager import DeviceManager

dm = DeviceManager()
run_seq.device_manager = dm

## 3. 运行电路并接收数据集

此单元是实际实验调用。每个电路执行 `reps` 次，然后产生一行概率数据。默认未开启 QST，所以四个电路对应四行。

`collect=True` 返回 `QDataset`；即使设为 `False`，程序仍然向 Data Vault 保存数据，只是不返回对象。

额外的 `reference_...` 参数通过 `**kwargs` 保存为元数据，记录电路路径和测量基的对应关系。

In [ ]:
reload(qc)  # 在完成第 0 节的源码修正后重新加载

ds = qc.run_circuit(
    device_configs=qubit_configs,
    wiring_configs=wiring_configs,
    opt_qubits=opt_qubits,
    read_qubits=read_qubits,
    opt_couplers=opt_couplers,
    circuit_path=[str(p) for p in circuit_paths],
    axes=["circuit", "QST_base"],
    reps=reps,
    nstate=2,
    measure_base=measure_base,
    QST=do_qst,
    cosine_env=False,
    read_delay=100 * ns,
    use_DD=False,
    data_vault_path=data_vault_path,
    collect=True,
    des="reference notebook",
    reference_circuit_paths=[str(p) for p in circuit_paths],
    reference_measure_base=measure_base,
    reference_QST=do_qst,
)

print("Data Vault 目录:", ds.path)
print("数据集编号:", ds.num)
print("数据集名称:", ds.fullName)
print("磁盘目录:", ds.parent_path)

## 4. 从返回对象获取概率表

`ds.data` 是由一维 NumPy 数组组成的列表。每行前面是扫描变量，后面是概率。

默认四比特列顺序为：`circuit, P0000, P0001, ..., P1111`。例如 `P0010` 表示 `read_qubits[2]` 测得 1，其他比特测得 0。若使用 X/Y 基，bitstring 表示基旋转后的测量结果。

这里用 dependent 的 label（如 `P0000`）作为列名，避免所有概率列都叫 `prob`。

In [ ]:
axis_names = [name for name, unit in ds.independents]
prob_cols = [label for category, label, unit in ds.dependents]

data = np.asarray(ds.data, dtype=float)
df = pd.DataFrame(data, columns=axis_names + prob_cols)
df["circuit"] = df["circuit"].astype(int)
if "QST_base" in df:
    df["QST_base"] = df["QST_base"].astype(int)

probabilities = df[prob_cols].to_numpy()
saved_params = dict(ds.params)
analysis_qubits = list(saved_params["read_qubits"])
analysis_reps = int(saved_params["reps"])
analysis_base = str(saved_params.get("reference_measure_base", measure_base))

print("数组形状:", data.shape)
display(df)
display(df[axis_names].assign(probability_sum=probabilities.sum(axis=1)))

## 5. 联合概率柱状图

每个概率是该状态出现次数 / 实际测量次数。下面的误差条采用独立重复测量假设下的二项分布标准误近似 `sqrt(p(1-p)/reps)`，不包含读出标定误差，也不是置信区间。

QST 数据需同时指定 `circuit` 和 `QST_base`，不能把不同测量基的概率混在一起。

In [ ]:
selected_circuit = 0
selected_qst_base = 0  # 仅对 QST 数据生效

selection = df.loc[df["circuit"] == selected_circuit]
if "QST_base" in df:
    selection = selection.loc[selection["QST_base"] == selected_qst_base]
row = selection.iloc[0]  # 有额外配置扫描时，还需先筛选对应扫描条件

p = row[prob_cols].to_numpy(dtype=float)
stderr = np.sqrt(np.clip(p * (1 - p), 0, None) / analysis_reps)

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar([c[1:] for c in prob_cols], p, yerr=stderr, capsize=2)
ax.set(xlabel="Measured bitstring", ylabel="Probability",
       title=f"Circuit {selected_circuit}")
ax.tick_params(axis="x", rotation=60)
ax.set_ylim(0, max(0.1, float(np.max(p + stderr)) * 1.15))
fig.tight_layout()
plt.show()

## 6. 从联合概率计算单比特概率和关联量

对于 Z 基：`P(q=1)` 是单比特激发概率，`<Z_i> = 1 - 2 P(q_i=1)`，`<Z_i Z_j>` 是两比特奇偶关联。

X/Y 基旋转遵循当前代码：X 基用 `-Y/2`，Y 基用 `X/2`。在门标定与该约定一致时，同样的 ±1 加权对应相应的 Pauli 期望值。X/Y 基的 bit=1 概率不应解释为原始能量基激发概率。

这些结果直接来自实验频率，没有应用读出误差校正。

In [ ]:
n = len(analysis_qubits)
bitstrings = [name.removeprefix("P") for name in prob_cols]
bits = np.array([[int(bit) for bit in state] for state in bitstrings])
signs = 1 - 2 * bits

P = df[prob_cols].to_numpy(dtype=float)
summary = df[axis_names].copy()

if "QST_base" in df:
    basis_strings = ["".join(v) for v in product("XYZ", repeat=n)]
    summary["basis"] = [basis_strings[int(i)] for i in df["QST_base"]]
else:
    summary["basis"] = analysis_base * n

for i, qubit in enumerate(analysis_qubits):
    summary[f"P({qubit}=1)"] = P @ bits[:, i]
    summary[f"<sigma_{qubit}>"] = P @ signs[:, i]

# 示例：前两个读出比特的关联；sigma 的具体轴由 basis 列决定
if n >= 2:
    summary["<sigma_0 sigma_1>"] = P @ (signs[:, 0] * signs[:, 1])

# 所有读出比特的奇偶关联
summary["<parity>"] = P @ np.prod(signs, axis=1)
display(summary)

In [ ]:
# 各电路的单比特 bit=1 概率；QST 时仅展示一个测量基
plot_df = summary
if "QST_base" in plot_df:
    plot_df = plot_df.loc[plot_df["QST_base"] == selected_qst_base]

fig, ax = plt.subplots(figsize=(8, 4))
for qubit in analysis_qubits:
    ax.plot(plot_df["circuit"], plot_df[f"P({qubit}=1)"],
            "o-", label=qubit)
ax.set(xlabel="Circuit index", ylabel="Marginal P(bit=1)", ylim=(0, 1))
ax.set_xticks(sorted(plot_df["circuit"].unique()))
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## 7. 已有数据：从磁盘重新读取（替代第 2–4 节）

项目已有的读数方式是 `labcodes.fileio.read_labrad`。Data Vault 在磁盘使用 `.csv` 保存数值、同名 `.ini` 保存列定义与参数。根目录来自 LabRAD Registry 的 `Servers / Data Vault / Repository`。

还保留 `ds` 时可用下面的目录和编号。重新打开 notebook 时，换成之前记录下来的值。本节产生的 `df / prob_cols / axis_names` 可以继续用于第 5–6 节。

`labcodes` 通常把概率列命名为 `prob_P0000` 等，下面将其统一为 `P0000` 等。

In [ ]:
from labcodes import fileio
import re

# 若是在新 kernel 中读取历史数据，将这两行换成实际磁盘目录与编号：
DATA_DIR = ds.parent_path
DATA_ID = ds.num

logf = fileio.read_labrad(DATA_DIR, id=DATA_ID)
df = logf.df.copy()
saved_params = dict(logf.conf["parameter"])

df = df.rename(columns={
    col: col.removeprefix("prob_")
    for col in df.columns
    if re.fullmatch(r"prob_P[01]+", str(col))
})

# 显式按二进制顺序排列，避免依赖显示顺序
prob_cols = sorted(
    [col for col in df.columns if re.fullmatch(r"P[01]+", str(col))],
    key=lambda col: int(col[1:], 2),
)
axis_names = [col for col in df.columns if col not in prob_cols]
df["circuit"] = df["circuit"].astype(int)
if "QST_base" in df:
    df["QST_base"] = df["QST_base"].astype(int)

analysis_qubits = list(saved_params["read_qubits"])
analysis_reps = int(saved_params["reps"])
# 历史数据可能没有 reference_measure_base，需要根据原实验填写
analysis_base = str(saved_params.get("reference_measure_base", "Z"))

display(df)
print("读出比特顺序:", analysis_qubits)

也可以直接通过 LabRAD 服务读取，不要求客户端能访问服务器的磁盘。以下为独立替代示例，替换逻辑目录和编号后使用；`dv.variables()` 返回列定义。

In [ ]:
# 可选：取消注释并填入自己的目录和编号
# import labrad
#
# with labrad.connect() as cxn:
#     dv = cxn.data_vault
#     dv.cd(["", "N36X21", "20260909_run_circuit_example"])
#     dv.open(123)  # 替换为数据集编号
#     indeps, deps = dv.variables()
#     stored_data = np.asarray(dv.get(), dtype=float)
#     stored_params = dict(dv.get_parameters())
#
# stored_columns = [v[0] for v in indeps] + [v[1] for v in deps]
# stored_df = pd.DataFrame(stored_data, columns=stored_columns)
# display(stored_df)

## 8. 导出带表头的 CSV 和分析所需元数据

本节将当前选用的 `df` 及分析结果导出为普通 CSV，便于用 pandas 或 Excel 读取。原始 Data Vault 文件保持原样。

如只做了历史取数，还未运行第 6 节，可以先注释掉 `summary.to_csv(...)`。

In [ ]:
export_dir = PROJECT_ROOT / "test" / "run_circuit_exports"
export_dir.mkdir(parents=True, exist_ok=True)

# 使用自己的数据集编号，避免不同实验混用文件名
export_id = int(DATA_ID if "DATA_ID" in globals() else ds.num)
prefix = f"dataset_{export_id:05d}"

df.to_csv(export_dir / f"{prefix}_probabilities.csv",
          index=False, encoding="utf-8-sig")
summary.to_csv(export_dir / f"{prefix}_observables.csv",
               index=False, encoding="utf-8-sig")

metadata = {
    "dataset_id": export_id,
    "read_qubits": analysis_qubits,
    "reps": analysis_reps,
    "axis_columns": axis_names,
    "probability_columns": prob_cols,
    "measure_base": saved_params.get("reference_measure_base", analysis_base),
    "QST": saved_params.get("reference_QST", "QST_base" in df),
    "circuit_paths": saved_params.get("reference_circuit_paths", []),
}
(export_dir / f"{prefix}_metadata.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)
print("导出目录:", export_dir)

## 9. 可选：采集 QST 测量基

要采集 QST，将第 2 节的 `do_qst` 改为 `True`，再执行实验调用。在本示例 `read_qubits == opt_qubits` 且顺序相同的前提下，N 比特有 `3**N` 个基。

四比特、四个电路共 `4 * 3**4 = 324` 个扫描点；每点 `reps` 次。表的列为 `circuit, QST_base, P0000, ..., P1111`。第 5–6 节已经支持按测量基筛选和添加基标签。

这里仅演示读取各基概率及 Pauli 关联，没有进行密度矩阵重建。

In [ ]:
# 不发起新实验：仅在当前 df 确实是 QST 数据时显示基索引映射
if "QST_base" in df:
    basis_strings = ["".join(v) for v in product("XYZ", repeat=len(analysis_qubits))]
    basis_table = pd.DataFrame({
        "QST_base": range(len(basis_strings)),
        "basis": basis_strings,
    })
    display(basis_table)
    display(df.merge(basis_table, on="QST_base").head())
else:
    print("当前数据未包含 QST_base；这是固定测量基的结果。")

## 源码定位

- `qcontrol/experiment/circuits.py`：`run_circuit`，执行电路、分析概率并返回数据集。
- `qcontrol/experiment/run_seq.py`：`run_seq`，返回按比特索引的 IQ 数组，shape 为 `(测量次数, 1, 2)`。
- `qcontrol/experiment/experiments.py`：`analyze_probs`，二态判别与联合计数。
- `qcontrol/experiment/utils/dataset.py`：`QDataset.data` 和 `save_to_dataset`。
- `qcontrol/server/data_server.py`：保存到 Data Vault，记录编号、名称和磁盘目录。
- `qcontrol/experiment/lsingQS/plot_lsing.py`：项目现有的 `fileio.read_labrad` 使用示例。

上述路径均相对于本项目的 `D:\qcontrol_gd\qcontrol`。